# RAG System Evaluation - Stage 1

This notebook evaluates the RAG (Retrieval-Augmented Generation) system for:

1. **Retrieval Accuracy** - Recall@K, Precision@K, MRR
2. **Performance** - Response times
3. **End-to-End Quality** - Answer quality assessment

**Test Dataset:** 10 queries covering policy, pricing, amenities, and restrictions

## 1. Setup

In [ ]:
import os
import sys
from dotenv import load_dotenv

# Add parent directory to path
sys.path.append(os.path.abspath('..'))

# Load environment variables
load_dotenv()

print(" Setup complete")

In [ ]:
from chatbot.main import ParkingChatbot
from chatbot.evaluation import RAGEvaluator
from chatbot.nodes import _get_vector_store, _get_compression_retriever

print(" Imports successful")

## 2. Initialize Components

In [ ]:
print("Initializing components...")

# Initialize chatbot
chatbot = ParkingChatbot()

# Get retrieval components
vector_store = _get_vector_store()
retriever = _get_compression_retriever()

# Initialize evaluator
evaluator = RAGEvaluator()

print(f" Initialized with {len(evaluator.test_cases)} test cases")

## 3. View Test Cases

In [ ]:
import pandas as pd

# Display test cases
test_df = pd.DataFrame([{
    "Query": tc["query"],
    "Category": tc["category"],
    "Keywords": ", ".join(tc["relevant_keywords"][:3]) + "..."
} for tc in evaluator.test_cases])

print("Test Cases:")
display(test_df)

## 4. Retrieval Evaluation

Tests the RAG retrieval pipeline:
- Vector search (Milvus)
- Reranking (Cross-Encoder)
- Metrics: Recall@K, Precision@K, MRR

In [ ]:
print("Running retrieval evaluation...\n")

retrieval_metrics, retrieval_results = evaluator.evaluate_retrieval(
    vector_store=vector_store,
    retriever=retriever,
    k_values=[3, 10]
)

print("\n" + "="*70)
print("RETRIEVAL EVALUATION RESULTS")
print("="*70)

print(f"\nAverage Recall@3:     {retrieval_metrics['avg_recall_at_3']:.2%}")
print(f"Average Precision@3:  {retrieval_metrics['avg_precision_at_3']:.2%}")
print(f"Average Recall@10:    {retrieval_metrics['avg_recall_at_10']:.2%}")
print(f"Average Precision@10: {retrieval_metrics['avg_precision_at_10']:.2%}")
print(f"Average MRR:          {retrieval_metrics['avg_mrr']:.3f}")
print(f"Average Time:         {retrieval_metrics['avg_retrieval_time']:.3f}s")

print("\n" + "="*70)

### Detailed Retrieval Results

In [ ]:
retrieval_df = pd.DataFrame([{
    "Query": r.query[:50] + "...",
    "Recall@3": f"{r.recall_at_3:.1%}",
    "Precision@3": f"{r.precision_at_3:.1%}",
    "MRR": f"{r.mrr:.3f}",
    "Time (s)": f"{r.retrieval_time:.3f}"
} for r in retrieval_results])

display(retrieval_df)

## 5. End-to-End Evaluation

Tests the complete chatbot system:
- Full conversation flow
- Response quality
- Total response time

In [ ]:
print("Running end-to-end evaluation...\n")
print("This may take 30-60 seconds as it tests all 10 queries...\n")

e2e_metrics, e2e_results = evaluator.evaluate_end_to_end(chatbot)

print("\n" + "="*70)
print("END-TO-END EVALUATION RESULTS")
print("="*70)

print(f"\nAverage Response Time:  {e2e_metrics['avg_response_time']:.3f}s")
print(f"Average Quality Score:  {e2e_metrics['avg_quality_score']:.2%}")
print(f"\nResponse Quality Distribution:")
print(f"  Excellent:  {e2e_metrics['excellent_responses']}/{e2e_metrics['total_queries']}")
print(f"  Good:       {e2e_metrics['good_responses']}/{e2e_metrics['total_queries']}")
print(f"  Fair:       {e2e_metrics['fair_responses']}/{e2e_metrics['total_queries']}")
print(f"  Poor:       {e2e_metrics['poor_responses']}/{e2e_metrics['total_queries']}")

print("\n" + "="*70)

### Detailed End-to-End Results

In [ ]:
e2e_df = pd.DataFrame([{
    "Query": r.query[:50] + "...",
    "Quality": r.response_quality,
    "Time (s)": f"{r.total_time:.3f}"
} for r in e2e_results])

display(e2e_df)

### Sample Responses

In [ ]:
# Show a few sample Q&A pairs
print("Sample Responses:\n")
print("="*70)

for i, test_case in enumerate(evaluator.test_cases[:3], 1):
    chatbot.reset()
    query = test_case["query"]
    response = chatbot.chat(query)
    
    print(f"\n[{i}] Query: {query}")
    print(f"\nResponse: {response}")
    print("\n" + "="*70)

## 6. Generate Report

In [ ]:
report = evaluator.generate_report(
    retrieval_metrics=retrieval_metrics,
    retrieval_results=retrieval_results,
    e2e_metrics=e2e_metrics,
    e2e_results=e2e_results,
    output_file="../../evaluation_report.json"
)

print(" Report generated: evaluation_report.json")
print("\nSummary:")
print(f"  Retrieval Performance: {report['summary']['retrieval_performance']}")
print(f"  Response Quality:      {report['summary']['response_quality']}")
print(f"  Response Time:         {report['summary']['response_time']}")

## 7. Visualizations

In [ ]:
import matplotlib.pyplot as plt

# Plot 1: Retrieval Metrics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Recall and Precision
metrics = ['Recall@3', 'Precision@3', 'Recall@10', 'Precision@10']
values = [
    retrieval_metrics['avg_recall_at_3'],
    retrieval_metrics['avg_precision_at_3'],
    retrieval_metrics['avg_recall_at_10'],
    retrieval_metrics['avg_precision_at_10']
]

axes[0].bar(metrics, values, color=['#2E86AB', '#A23B72', '#2E86AB', '#A23B72'])
axes[0].set_ylabel('Score')
axes[0].set_title('Retrieval Accuracy Metrics')
axes[0].set_ylim([0, 1])
axes[0].grid(axis='y', alpha=0.3)

# Response Quality Distribution
quality_labels = ['Excellent', 'Good', 'Fair', 'Poor']
quality_counts = [
    e2e_metrics['excellent_responses'],
    e2e_metrics['good_responses'],
    e2e_metrics['fair_responses'],
    e2e_metrics['poor_responses']
]

axes[1].pie(quality_counts, labels=quality_labels, autopct='%1.1f%%',
            colors=['#06A77D', '#2E86AB', '#F77F00', '#D62828'])
axes[1].set_title('Response Quality Distribution')

plt.tight_layout()
plt.show()

# Plot 2: Response Times
fig, ax = plt.subplots(figsize=(10, 5))

queries = [r.query[:30] + "..." for r in e2e_results]
times = [r.total_time for r in e2e_results]

ax.barh(queries, times, color='#2E86AB')
ax.set_xlabel('Response Time (seconds)')
ax.set_title('Response Time by Query')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

 **Evaluation Complete!**

**Key Findings:**
- Retrieval system evaluated with Recall@K and Precision@K metrics
- End-to-end response quality assessed
- Performance benchmarks established
- Detailed report saved to `evaluation_report.json`

**Next Steps:**
- Review low-scoring queries
- Fine-tune retrieval parameters if needed
- Monitor performance in production
- Expand test dataset as system evolves